In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# reset defalult plotting values
plt.rcParams['figure.figsize'] = (9, 5)
plt.rc('font', family='sans-serif')
plt.rc('axes', labelsize=14)
plt.rc('axes', labelweight='bold')
plt.rc('axes', titlesize=16)
plt.rc('axes', titleweight='bold')

# ASTR 350: Astronomical Techniques
# Lecture 15


### Prof. Robert Quimby <br> March 10, 2026

&copy; 2026 Robert Quimby

## Fitting data to models

## Least-Squares Fitting

Tutorial
  - [notebook](../tutorials/least-squares.ipynb)
  - [video](https://youtu.be/6XYknfueYkc) (34 min)

## Least-Squares Fitting with a Coordinate Shift

Tutorial
- [notebook](../tutorials/lsq-fits.with.a.shift.ipynb)
- [video](https://youtu.be/MexPmPyg7UI) (19 min)



## Fitting the data with a model

In [ ]:
xs = [1.3, 3.4, 6.4]
ys = [2.1, 5.9, 13.5]
plt.plot(xs, ys, 'ro');

Assume that the values of the independent variable, $x$, can be used to predict the value of the dependent variable, $y$, and that measurements of the dependent variable include random uncertainty:

$$y = f(x) + \epsilon$$

where $\epsilon$ is the random noise in the measurement. 

We can then choose a functional form for $f(x)$, the relationship between $x$ and $y$, such as the linear model, $f(x) = mx + b$ (other models may be possible!). 

In [ ]:
# show a linear model
m = ????
b = ????
modelx = np.linspace(0, 8, 10)
modely = m * modelx + b

plt.plot(xs, ys, 'ro')
plt.plot(modelx, modely, '--');

## If the system is overdetermined (more data than parameters)

- There may not be a set of parameters that solves all equations
- So instead we try to find the "best" set of parameters

## What does "best" parameters mean?

* **IF** we can assume that the deviations from our ideal model are random and drawn from a Gaussian distribution
* **THEN** the "best" parameters are the set that minimizes the sum of the squares of the residuals

For the linear model, $y = mx + b$, the sum of the squares of the residuals as a function of $m$ and $b$ is:
$$S(m, b) = \Sigma [y_i - (mx_i + b)]^2$$

where the $y_i$ are the observed $y$ values and $mx_i + b$ are the predicted $y$ values given the model.

For the 3 points in our example above we have:
$$S(m, b) =  (2.1 - (1.3m + b))^2 + (5.9 - (3.4m + b))^2 + (13.5 - (6.4m + b))^2$$

which simplifies to:

$$ S(m, b) = 54.21 m^2 + 22.2  m  b - 218.38  m + 3 b^2 - 43  b + 221.47$$

## To find the best model parameters we just minimize $S(m, b)$

$$
\begin{align}
\delta S / \delta m  & = & 108.42 m + 22.2 b - 218.38 & = 0 \\
\delta S / \delta b  & = & 6 b + 22.2 m - 43 & = 0 \\
\end{align}
$$


In [ ]:
m = 494. / 219.
b = (43 - 22.2 * m) / 6
print(f"the best-fit parameters are m={m:.3f}, b={b:.3f}")

In [ ]:
# now we can plot the best fit line with our data
modely = m * modelx + b
plt.plot(xs, ys, 'ro')
plt.plot(modelx, modely, '--');

## That was too much work; lets make it easier...

We can take the three equations:

$$
\begin{align}
2.1   &  =  1.3m + b\\
5.9   &  =  3.4m + b\\
13.5  &  =  6.4m + b\\
\end{align}
$$

and turn them into a single matrix equation:

$$ Y = Xp $$

$Y = 
\left[ \begin{array}{c}
2.1  \\
5.9  \\
13.5  \end{array} \right] $, 
$X = 
\left[ \begin{array}{cc}
1.3 & 1 \\
3.4 & 1 \\
6.4 & 1 \end{array} \right] 
$, and $p = \left[ \begin{array}{c}
m \\
b \end{array} \right] $

## Numpy Matrix objects

In [ ]:
# if we pass a list of lists to np.matrix, we get a 2D matrix
X = np.matrix([[xi, 1] for xi in xs])
Y = np.matrix([[yi] for yi in ys])

In [ ]:
print(X)
print()
print(Y)

In [ ]:
print(f"X is a {X.shape} matrix")
print(f"Y is a {Y.shape} matrix")

## numpy.array vs. numpy.matrix

In [ ]:
# matrix operations
X.T * X, (X.T * X).I

In [ ]:
# You can do array math with numpy.array (just not as pretty)
A = np.array(X)
A.T @ A, np.linalg.inv(A.T @ A)

## Now we can just solve for $p$

$$ Xp = Y $$

$$ X^T X p = X^T Y $$

In [ ]:
(X.T * X).shape

$$ (X^T X)^{-1} (X^T X) p = (X^T X)^{-1} X^T Y $$

In [ ]:
(X.T * X).I * (X.T * X)

$$ p = (X^T X)^{-1} X^T Y $$

In [ ]:
p = ????
p

#### Notice we get the same values for $p$ as before!

## The product, $Xp$, gives the $y$ values _predicted_ by the model

In [ ]:
modelY = ????

In [ ]:
plt.plot(xs, ys, 'ro')

plt.plot(X[:, 0], modelY, 'bx', ms=10, mew=3);
plt.plot(X[:, 0], modelY, '--');

## Uncertainty in the $y$ values

- Recall that least-squares fitting assumes the data - model residuals follow a random, Gaussian distribution
- If this is valid then we can use the residuals to estimate the uncertainty in the best-fit parameters

In [ ]:
# residuals from the model
dY = Y - modelY

# standard deviation of the residuals
sample_var = dY.var(ddof=p.shape[0])

## How does uncertainty in $y$ map to uncertainty in $m$ and $b$?

$$Y = Xp$$

So uncertainty in the parameters, $p$, must be related to uncertainty in $Y$ by the values in the matrix $X$

In [ ]:
X

In [ ]:
((X.T * X).I)

In [ ]:
# uncertainty in the model parameters
pvar = sample_var * np.diagonal( (X.T * X).I )

In [ ]:
# best fit parameters
m, b = p.A1
msig, bsig = np.sqrt(pvar)

# now for the parameter errors
print("m is {:.3f} +/- {:.3f}".format(m, msig))
print("b is {:.3f} +/- {:.3f}".format(b, bsig))

## Are you getting this?

Do not download until told to do so:
- https://classroom.github.com/a/od7J5VlP